In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Math, HTML

# ============================================================
# FOURIER TRANSFORM METHOD FOR A DIFFERENTIAL EQUATION
#
# y'(t) + y(t) = (1/2) exp(-|t|)
#
# Fourier convention:
#
# F(omega) = integral f(t) exp(-i*omega*t) dt
#
# f(t) = 1/(2*pi) integral F(omega) exp(i*omega*t) d omega
#
# Only the original problem and the GENERAL Fourier derivative
# property are supplied. All problem-specific expressions are
# produced symbolically by SymPy.
# ============================================================

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div style="
    width:1050px;
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.50;
    margin-bottom:12px;
">

<div style="
    font-size:20px;
    font-weight:bold;
    color:#6f3fa0;
    margin-bottom:8px;
">
Fourier Transform Method for a Differential Equation
</div>

<div style="margin-bottom:5px;">
This notebook solves the first-order ordinary differential equation
y'(t)+y(t)=½e<sup>−|t|</sup> by the Fourier-transform method.
</div>

<div style="margin-bottom:5px;">
Only the original differential equation and the general Fourier
property for differentiation are supplied. All quantities specific
to this problem are subsequently calculated symbolically by SymPy.
</div>

<div style="margin-bottom:5px;">
Because the forcing term contains |t|, its defining Fourier integral
is split at t=0. If SymPy returns several Piecewise branches, the
first convergent branch appropriate to real ω is retained.
</div>

<div>
The transformed algebraic equation, its poles and residues, and the
final time-domain solution are all generated symbolically.
</div>

</div>
"""))

# ============================================================
# SYMBOLS
# ============================================================

t = sp.symbols('t', real=True)
omega = sp.symbols('omega', real=True)
z = sp.symbols('z')
I = sp.I

y = sp.Function('y')
Yw = sp.Function('Y')

# ============================================================
# HELPER FUNCTION
#
# For Piecewise:
#
# expr.args[0]     -> (first expression, first condition)
# expr.args[0][0]  -> first expression
# ============================================================

def first_piece(expr):
    if isinstance(expr, sp.Piecewise):
        return sp.simplify(expr.args[0][0])
    return sp.simplify(expr)

# ============================================================
# ORIGINAL PROBLEM
# ============================================================

lhs_time = sp.diff(y(t), t) + y(t)
rhs_time = sp.Rational(1, 2) * sp.exp(-sp.Abs(t))
original_equation = sp.Eq(lhs_time, rhs_time)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:5px;
    margin-bottom:4px;
">
Original differential equation
</div>
"""))

display(Math(sp.latex(original_equation)))

# ============================================================
# GENERAL FOURIER DIFFERENTIATION PROPERTY
#
# This is the theoretical rule used by the method.
# The expression itself is constructed symbolically.
# ============================================================

transformed_derivative = I * omega * Yw(omega)

derivative_property = sp.Eq(
    sp.Symbol(
        r'\mathcal{F}\{y^{\prime}(t)\}'
    ),
    transformed_derivative
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:14px;
    margin-bottom:4px;
">
General differentiation property
</div>
"""))

display(
    Math(
        r'\mathcal{F}\left\{\dfrac{dy(t)}{dt}\right\}'
        r'='
        +
        sp.latex(transformed_derivative)
    )
)

# ============================================================
# SPLIT THE FORCING TERM SYMBOLICALLY
#
# exp(-|t|) becomes:
#
# exp(t)   for t < 0
# exp(-t)  for t > 0
# ============================================================

forcing_negative = (
    sp.Rational(1, 2)
    * sp.exp(t)
    * sp.exp(-I * omega * t)
)

forcing_positive = (
    sp.Rational(1, 2)
    * sp.exp(-t)
    * sp.exp(-I * omega * t)
)

# ============================================================
# SYMBOLIC INTEGRATION
# ============================================================

negative_raw = sp.integrate(
    forcing_negative,
    (t, -sp.oo, 0)
)

positive_raw = sp.integrate(
    forcing_positive,
    (t, 0, sp.oo)
)

# ============================================================
# REMOVE THE UNNECESSARY PIECEWISE BRANCHES
# ============================================================

negative_integral = first_piece(
    negative_raw
)

positive_integral = first_piece(
    positive_raw
)

# ============================================================
# DISPLAY SYMBOLIC INTEGRALS
# ============================================================

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:14px;
    margin-bottom:4px;
">
Symbolic Fourier transform of the forcing term
</div>
"""))

display(
    Math(
        r'\mathcal{F}\left\{\dfrac{1}{2}e^{-|t|}\right\}'
        r'='
        r'\dfrac{1}{2}\int_{-\infty}^{0}'
        r'e^{t}e^{-i\omega t}\,dt'
        r'+'
        r'\dfrac{1}{2}\int_{0}^{\infty}'
        r'e^{-t}e^{-i\omega t}\,dt'
    )
)

display(
    Math(
        r'\dfrac{1}{2}\int_{-\infty}^{0}'
        r'e^{t}e^{-i\omega t}\,dt'
        r'='
        +
        sp.latex(negative_integral)
    )
)

display(
    Math(
        r'\dfrac{1}{2}\int_{0}^{\infty}'
        r'e^{-t}e^{-i\omega t}\,dt'
        r'='
        +
        sp.latex(positive_integral)
    )
)

# ============================================================
# COMPLETE SYMBOLIC TRANSFORM OF THE FORCING TERM
# ============================================================

forcing_transform = sp.factor(
    sp.simplify(
        negative_integral
        +
        positive_integral
    )
)

display(
    Math(
        r'\mathcal{F}\left\{\dfrac{1}{2}e^{-|t|}\right\}'
        r'='
        +
        sp.latex(forcing_transform)
    )
)

# ============================================================
# BUILD THE TRANSFORMED DIFFERENTIAL EQUATION SYMBOLICALLY
#
# F{y'} + F{y} = F{forcing}
#
# ============================================================

transformed_lhs = (
    transformed_derivative
    +
    Yw(omega)
)

transformed_equation = sp.Eq(
    transformed_lhs,
    forcing_transform
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:14px;
    margin-bottom:4px;
">
Transformed differential equation
</div>
"""))

display(
    Math(
        sp.latex(
            transformed_equation
        )
    )
)

# ============================================================
# SOLVE SYMBOLICALLY FOR Y(omega)
# ============================================================

Y_solution_list = sp.solve(
    transformed_equation,
    Yw(omega)
)

Y_expression = sp.simplify(
    Y_solution_list[0]
)

display(
    Math(
        r'Y(\omega)='
        +
        sp.latex(Y_expression)
    )
)

# ============================================================
# FACTOR THE TRANSFORMED SOLUTION SYMBOLICALLY
# ============================================================

Y_factored = sp.factor(
    Y_expression,
    extension=I
)

display(
    Math(
        r'Y(\omega)='
        +
        sp.latex(Y_factored)
    )
)

# ============================================================
# CONSTRUCT THE INVERSE-TRANSFORM INTEGRAND
# ============================================================

Y_z = sp.simplify(
    Y_expression.subs(
        omega,
        z
    )
)

inverse_integrand = sp.simplify(
    Y_z
    *
    sp.exp(I * z * t)
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:14px;
    margin-bottom:4px;
">
Inverse Fourier transform
</div>
"""))

display(
    Math(
        r'y(t)='
        r'\dfrac{1}{2\pi}'
        r'\int_{-\infty}^{\infty}'
        +
        sp.latex(inverse_integrand)
        +
        r'\,dz'
    )
)

# ============================================================
# FIND POLES SYMBOLICALLY
# ============================================================

denominator_z = sp.factor(
    sp.denom(
        sp.cancel(
            Y_z
        )
    )
)

poles = sp.solve(
    denominator_z,
    z
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:14px;
    margin-bottom:4px;
">
Poles
</div>
"""))

display(
    Math(
        r'\mathrm{Poles}='
        +
        sp.latex(poles)
    )
)

# ============================================================
# DETERMINE POLES IN EACH HALF-PLANE SYMBOLICALLY
# ============================================================

upper_poles = [
    p for p in poles
    if sp.im(p).is_positive
]

lower_poles = [
    p for p in poles
    if sp.im(p).is_negative
]

display(
    Math(
        r'\mathrm{Upper\ half\!-\!plane\ poles}='
        +
        sp.latex(upper_poles)
    )
)

display(
    Math(
        r'\mathrm{Lower\ half\!-\!plane\ poles}='
        +
        sp.latex(lower_poles)
    )
)

# ============================================================
# SYMBOLIC RESIDUES FOR t > 0
#
# Upper-half-plane contour.
# ============================================================

upper_residues = [
    sp.simplify(
        sp.residue(
            inverse_integrand,
            z,
            pole
        )
    )
    for pole in upper_poles
]

upper_residue_sum = sp.simplify(
    sum(
        upper_residues,
        sp.Integer(0)
    )
)

# Inverse-transform factor:
#
# (1/2pi)(2pi i) = i
# ============================================================

y_positive = sp.factor(
    sp.simplify(
        I
        *
        upper_residue_sum
    )
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:16px;
    margin-bottom:4px;
">
Case t &gt; 0
</div>
"""))

for pole, residue in zip(
    upper_poles,
    upper_residues
):
    display(
        Math(
            r'\operatorname{Res}_{z='
            +
            sp.latex(pole)
            +
            r'}f(z)='
            +
            sp.latex(residue)
        )
    )

display(
    Math(
        r'y(t)='
        +
        sp.latex(y_positive)
        +
        r',\;t>0'
    )
)

# ============================================================
# SYMBOLIC RESIDUES FOR t < 0
#
# Lower-half-plane contour is clockwise.
#
# (1/2pi)(-2pi i) = -i
# ============================================================

lower_residues = [
    sp.simplify(
        sp.residue(
            inverse_integrand,
            z,
            pole
        )
    )
    for pole in lower_poles
]

lower_residue_sum = sp.simplify(
    sum(
        lower_residues,
        sp.Integer(0)
    )
)

y_negative = sp.factor(
    sp.simplify(
        -I
        *
        lower_residue_sum
    )
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:16px;
    margin-bottom:4px;
">
Case t &lt; 0
</div>
"""))

for pole, residue in zip(
    lower_poles,
    lower_residues
):
    display(
        Math(
            r'\operatorname{Res}_{z='
            +
            sp.latex(pole)
            +
            r'}f(z)='
            +
            sp.latex(residue)
        )
    )

display(
    Math(
        r'y(t)='
        +
        sp.latex(y_negative)
        +
        r',\;t<0'
    )
)

# ============================================================
# VALUE AT t = 0
# ============================================================

left_limit = sp.simplify(
    sp.limit(
        y_negative,
        t,
        0,
        dir='-'
    )
)

right_limit = sp.simplify(
    sp.limit(
        y_positive,
        t,
        0,
        dir='+'
    )
)

same_limit = sp.simplify(
    left_limit
    -
    right_limit
)

if same_limit == 0:
    y_zero = left_limit
else:
    y_zero = sp.nan

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:16px;
    margin-bottom:4px;
">
Continuity at t = 0
</div>
"""))

display(
    Math(
        r'\lim_{t\to0^-}y(t)='
        +
        sp.latex(left_limit)
    )
)

display(
    Math(
        r'\lim_{t\to0^+}y(t)='
        +
        sp.latex(right_limit)
    )
)

# ============================================================
# FINAL SYMBOLIC SOLUTION
# ============================================================

final_solution = sp.Piecewise(
    (
        y_negative,
        t < 0
    ),
    (
        y_zero,
        sp.Eq(t, 0)
    ),
    (
        y_positive,
        t > 0
    )
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:19px;
    font-weight:bold;
    color:#16802b;
    margin-top:18px;
    margin-bottom:6px;
">
Final symbolic solution
</div>
"""))

display(
    Math(
        r'y(t)='
        +
        sp.latex(final_solution)
    )
)

# ============================================================
# DIRECT SYMBOLIC VERIFICATION
#
# Substitute the two branches into:
#
# y'(t) + y(t) - (1/2) exp(-|t|)
#
# separately for t > 0 and t < 0.
# ============================================================

forcing_positive_time = sp.Rational(1, 2) * sp.exp(-t)
forcing_negative_time = sp.Rational(1, 2) * sp.exp(t)

check_positive = sp.simplify(
    sp.diff(
        y_positive,
        t
    )
    +
    y_positive
    -
    forcing_positive_time
)

check_negative = sp.simplify(
    sp.diff(
        y_negative,
        t
    )
    +
    y_negative
    -
    forcing_negative_time
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:16px;
    margin-bottom:4px;
">
Symbolic verification
</div>
"""))

display(
    Math(
        r'y^{\prime}(t)+y(t)'
        r'-\dfrac{1}{2}e^{-t}'
        r'='
        +
        sp.latex(check_positive)
        +
        r',\;t>0'
    )
)

display(
    Math(
        r'y^{\prime}(t)+y(t)'
        r'-\dfrac{1}{2}e^{t}'
        r'='
        +
        sp.latex(check_negative)
        +
        r',\;t<0'
    )
)

# ============================================================
# NUMERICAL PLOT
# ============================================================

y_pos_num = sp.lambdify(
    t,
    y_positive,
    'numpy'
)

y_neg_num = sp.lambdify(
    t,
    y_negative,
    'numpy'
)

t_values = np.linspace(
    -5.0,
    5.0,
    2001
)

y_values = np.empty_like(
    t_values
)

negative_mask = (
    t_values < 0
)

positive_mask = (
    t_values > 0
)

zero_mask = np.isclose(
    t_values,
    0.0
)

y_values[negative_mask] = y_neg_num(
    t_values[negative_mask]
)

y_values[positive_mask] = y_pos_num(
    t_values[positive_mask]
)

y_values[zero_mask] = float(
    y_zero
)

fig, ax = plt.subplots(
    figsize=(8.0, 4.4)
)

ax.plot(
    t_values,
    y_values,
    linewidth=2.0
)

ax.axhline(
    0.0,
    linewidth=0.8
)

ax.axvline(
    0.0,
    linewidth=0.8
)

ax.set_title(
    'Symbolically Computed Solution y(t)',
    fontsize=14,
    fontweight='bold',
    color='#6f3fa0'
)

ax.set_xlabel('t')
ax.set_ylabel('y(t)')

ax.grid(
    True,
    linestyle=':',
    alpha=0.40
)

fig.subplots_adjust(
    left=0.10,
    right=0.97,
    top=0.88,
    bottom=0.14
)

plt.show()

# ============================================================
# INTERPRETATION
# ============================================================

display(HTML("""
<div style="
    width:1000px;
    padding:10px 13px;
    border:1px solid #d7c7e5;
    font-family:Arial, sans-serif;
    font-size:14px;
    line-height:1.55;
    box-sizing:border-box;
    margin-top:8px;
">

<div style="
    color:#6f3fa0;
    font-size:17px;
    font-weight:bold;
    margin-bottom:6px;
">
Interpretation
</div>

<div style="margin-bottom:6px;">
The only problem-specific input is the original differential equation.
The general differentiation property of the Fourier transform is used
as the theoretical rule of the method.
</div>

<div style="margin-bottom:6px;">
The Fourier transform of the forcing term is obtained from the defining
integrals. The transformed function Y(ω), its poles, all residues, and
both time-domain branches are then calculated symbolically.
</div>

<div style="margin-bottom:6px;">
The expressions returned by SymPy are also substituted back into the
original differential equation independently for t&lt;0 and t&gt;0.
A zero residual confirms the symbolic solution.
</div>

<div>
No coefficient, pole, residue, or final branch of the solution is
entered from the analytical answer.
</div>

</div>
"""))